In [ ]:
# 1. Environment Initialization and WRDS Connection
import wrds
import pandas as pd

# Connect to the WRDS Database
username = "warry"
db = wrds.Connection(wrds_username=username)

# 2. Table Structure Exploration
# View the Structure of the CSMAR Financial Main Table
financial_desc = db.describe_table(library="csmar", table="wrds_csmar_financial_master")
# Simplify Core Columns
financial_desc_small = financial_desc[["name", "type", "comment"]]
# Preview the First 20 Rows
print(" Preview of Core Fields in the CSMAR Financial Main Table ")
financial_desc_small.head(20)

: 

In [2]:
# 3. Define the analysis targets and core financial fields
stkcd = {
    "600887": "Yili Share",
    "600597": "Guangming Dairy"
}
date_condition = "2019-12-31"
financial_columns_list = ["stkcd", "accper", "typrep", "a001000000", "a002000000", "a003000000", "b001100000", "b002000000", "a001100000","a002100000", "a001200000"]
financial_columns = ", ".join(financial_columns_list)

# Core financial indicators
ids_today = [
    "a001000000",   # Total Assets
    "a002000000",   # Total Liabilities
    "a003000000",   # Total Shareholders' Equity
    "b001100000",   # Total Operating Revenue
    "b002000000"    # Net Profit
    "a001100000"    # Current Assets
    "a002100000"    # Current Liabilities
    "a001200000"    # Inventory
]

# 4. SQL queries
# Build SQL: Filter the annual report data from 2020 to 2024
financial_query = f"""
SELECT {financial_columns}
FROM csmar.wrds_csmar_financial_master
WHERE stkcd IN ('600887','600597')
AND accper > '{date_condition}'
AND typrep = 'A'
"""

df = db.raw_sql(financial_query, date_cols=["accper"])
df

,stkcd,accper,typrep,a001000000,a002000000,a003000000,b001100000,b002000000,a001100000,a002100000,a001200000
0,600597,2020-01-01,A,17637106805.0,10220165759.0,7416941046.0,22563236819.0,682452363.0,7205510710.0,8230772133.0,10431596095.0
1,600597,2020-03-31,A,17644279298.0,10650388984.0,6993890314.0,5134028633.0,150657386.0,6957052876.0,8884089802.0,10687226422.0
2,600597,2020-06-30,A,19485798699.0,11757912305.0,7727886394.0,12146010002.0,485788993.0,8347323320.0,8873006248.0,11138475379.0
3,600597,2020-09-30,A,20161643371.0,12265897267.0,7895746104.0,18724916802.0,592015503.0,8897481857.0,9328444757.0,11264161514.0
4,600597,2020-12-31,A,20309910295.0,11394646209.0,8915264086.0,25222715966.0,785141962.0,8976852381.0,9006812147.0,11333057914.0
5,600597,2021-01-01,A,20167240356.0,11235090981.0,8932149375.0,25266056840.0,785380665.0,9010956628.0,9075195578.0,11156283728.0
6,600597,2021-03-31,A,19972169854.0,11228162758.0,8744007096.0,6985625527.0,85836264.0,8233698305.0,8547865675.0,11738471549.0
7,600597,2021-06-30,A,20115746164.0,11506539914.0,8609206250.0,14264084192.0,219700602.0,8318259324.0,8869012384.0,11797486840.0
8,600597,2021-09-30,A,20325110196.0,11676114130.0,8648996066.0,22056891118.0,350121548.0,7958764453.0,7767840062.0,12366345743.0
9,600597,2021-12-31,A,23450401026.0,13098498836.0,10351902190.0,29205992515.0,566893573.0,9178291798.0,8689140992.0,14272109228.0


In [3]:
# 5. Data Cleaning
# 5.1 Extracting Years
df["year"] = df["accper"].dt.year

# 5.2 Mapping Company Names
df["company"] = df["stkcd"].map(stkcd)

# 5.3 Calculation 
df["net_profit_margin"] = df["b002000000"]/df["b001100000"]
df["roa"] = df["b002000000"]/df["a001000000"]
df["roe"] = df["b002000000"]/df["a003000000"]
df["current_ratio"] = df["a001100000"]/df["a002100000"]
df["quick_ratio"] = (df["a001100000"]-df["a001200000"])/df["a002100000"]
df["debt_to_assets_ratio"] = df["a002000000"]/df["a001000000"]

# 5.4 Handling of Missing Values and Outliers
# Missing core indicators are directly deleted
df = df.dropna(subset=["net_profit_margin", "roa", "roe", "current_ratio", "quick_ratio","debt_to_assets_ratio"])
# De-duplication: Keep the latest data for the same year
df = df.drop_duplicates(subset=["company", "year"], keep="last")
# Sort by year 
df = df.sort_values(by=["company", "year"]).reset_index(drop=True)


# Verify the cleaning results
print(" Preview of the cleaned data ")
df[["company", "year", "net_profit_margin", "roa", "roe", "current_ratio", "quick_ratio","debt_to_assets_ratio"]]

 Preview of the cleaned data 


,company,year,net_profit_margin,roa,roe,current_ratio,quick_ratio,debt_to_assets_ratio
0,Guangming Dairy,2020,0.031128,0.038658,0.088067,0.996674,-0.261603,0.561039
1,Guangming Dairy,2021,0.01941,0.024174,0.054762,1.056294,-0.586228,0.558562
2,Guangming Dairy,2022,0.013863,0.015997,0.037083,0.923048,-0.431821,0.56862
3,Guangming Dairy,2023,0.03136,0.034282,0.072713,0.953159,-0.469474,0.528525
4,Guangming Dairy,2024,0.019849,0.021014,0.043456,0.870985,-0.397723,0.516424
5,Yili Share,2020,0.073271,0.099768,0.232503,0.81628,-0.413973,0.570895
6,Yili Share,2021,0.078955,0.08564,0.178968,1.158414,-0.038165,0.52148
7,Yili Share,2022,0.075653,0.07115,0.172103,0.98864,-0.129303,0.586584
8,Yili Share,2023,0.081505,0.067829,0.179418,0.902362,-0.167962,0.621948
9,Yili Share,2024,0.073102,0.05506,0.148466,0.740837,-0.227738,0.629141


In [4]:
# 6. Mapping
import tkinter as tk
import pandas as pd

# 6.1 Data Extraction
yili = df[df["company"] == "Yili Share"].sort_values("year").reset_index(drop=True)
guangming = df[df["company"] == "Guangming Dairy"].sort_values("year").reset_index(drop=True)
years = [2020, 2021, 2022, 2023, 2024]

# 6 indicators
indicators = [
    ("net_profit_margin", "Net Profit Margin Comparison", "Net Profit Margin(%)", True),
    ("roa", "ROA Comparison", "ROA(%)", True),
    ("roe", "ROE Comparison", "ROE(%)", True),
    ("current_ratio", "Current Ratio Comparison", "Current Ratio", False),
    ("quick_ratio", "Quick Ratio Comparison", "Quick Ratio", False),
    ("debt_to_assets_ratio", "Debt_to_assets Ratio Comparison", "Debt_to_assets Ratio(%)", True)
]

#  6.2 Window Settings
root = tk.Tk()
root.title("Yili Share VS Guangming Dairy 2020-2024 Financial Comparison")
root.geometry("1300x820")
root.protocol("WM_DELETE_WINDOW", root.quit)

# Subplot size
w, h = 380, 260
px, py = 30, 30 

#  6.3 Start plotting
for idx, (col, title, yname, is_pct) in enumerate(indicators):
    row = idx // 3
    col_pos = idx % 3

    # Subplot position calculation
    x = px + col_pos * (w + px)
    y = py + row * (h + py)

    # Create canvas
    canvas = tk.Canvas(root, width=w, height=h, bg="white")
    canvas.place(x=x, y=y)

    # Get data
    y_vals = yili[col].tolist()
    g_vals = guangming[col].tolist()

    # Percentage × 100, retain 2 decimal places
    if is_pct:
        y_vals = [round(v*100, 2) for v in y_vals]
        g_vals = [round(v*100, 2) for v in g_vals]

    # Y-axis range
    all_vals = y_vals + g_vals
    y_min = min(all_vals) * 0.9 if all_vals else 0
    y_max = max(all_vals) * 1.1 if all_vals else 1
    # Special handling for quick ratio to ensure both positive and negative values are displayed
    if col == "quick_ratio":
        y_min = min(all_vals) - 0.1
        y_max = max(all_vals) + 0.1
    y_rng = y_max - y_min if y_max != y_min else 1

    # Coordinate area
    xs, ys = 40, 220  # left lower
    xe, ye = 320, 30  # upper right
    x_step = (xe - xs) / 4  # 5 years → 4 parts

    # Coordinate axes 
    canvas.create_line(xs, ys, xe, ys, width=2)
    canvas.create_line(xs, ys, xs, ye, width=2)

    #  X-axis year
    for i, year in enumerate(years):
        xx = xs + i * x_step
        canvas.create_text(xx, ys+12, text=str(year), font=("Arial", 9))

    # Y-axis scale
    for i in range(6):
        val = y_min + (y_rng)*i/5
        yy = ys - ((val - y_min)/y_rng)*(ys - ye)
        # Quick ratio shows 2 decimal places, others show 1 decimal place
        fmt = ".2f" if col == "quick_ratio" else ".1f"
        canvas.create_text(xs-8, yy, text=f"{val:{fmt}}", font=("Arial", 8), anchor="e")

    # Draw lines: Yili (red)
    px0, py0 = None, None
    for i, v in enumerate(y_vals):
        xx = xs + i*x_step
        yy = ys - ((v-y_min)/y_rng)*(ys-ye)
        if px0 is not None:
            canvas.create_line(px0, py0, xx, yy, fill='red', width=2)
            canvas.create_oval(xx-3, yy-3, xx+3, yy+3, fill='red')
        px0, py0 = xx, yy

    # Draw lines: Guangming (blue)
    px0, py0 = None, None
    for i, v in enumerate(g_vals):
        xx = xs + i*x_step
        yy = ys - ((v-y_min)/y_rng)*(ys-ye)
        if px0 is not None:
            canvas.create_line(px0, py0, xx, yy, fill='blue', width=2)
            canvas.create_oval(xx-3, yy-3, xx+3, yy+3, fill='blue')
        px0, py0 = xx, yy

    # Title, legend
    canvas.create_text((xs+xe)//2, 15, text=title, font=("Arial", 11, "bold"))
    # Yili legend
    canvas.create_rectangle(325, 40, 340, 55, fill='red')
    canvas.create_text(345, 47, text='Yili', font=('Arial', 9), anchor='w')
    # Guangming legend
    canvas.create_rectangle(325, 60, 340, 75, fill='blue')
    canvas.create_text(345, 67, text='Guangming', font=('Arial', 9), anchor='w')

# Start tkinter main loop
root.mainloop()

In [ ]:
db.close

: 